# SR-PDS simulation -- full HPC run

Production-quality results for the dissertation. Run this on icelake
after `main_quick.ipynb` has verified the pipeline.

**Configuration**

- All 8 DGPs, all 7 estimators
- Headline: n=500, n_reps=500 for everything except sr_pds_cf which
  uses n_reps=100 (cross-fit is 5x more expensive per rep and the
  point of having it is to show match against same-data SR-PDS,
  which doesn't need MC SE 0.02 -- 0.05 is plenty)
- Trajectory: n in {30, 40, ..., 100, 200, 500, 1000, 2000, 5000} at
  n_reps=25 (no sr_pds_cf in the trajectory -- it lives at n=500 only)
- Large-n extension: n=10,000 on DGPs 1, 6, 7, 8 and n=50,000 on DGPs
  6, 7, 8, both at n_reps=25
- Seed-robustness: 10 PySR seeds on one DGP-6 dataset at n=500

**Expected wall-clock on a 76-core icelake-himem node:** ~4-8 hours
total. PySR is the bottleneck; everything else parallelises well.

**SLURM advice:**

```
#SBATCH --partition=icelake-himem
#SBATCH --nodes=1 --cpus-per-task=76
#SBATCH --time=10:00:00 --mem=256G
export JULIA_NUM_THREADS=4
export OMP_NUM_THREADS=4
```

The `JULIA_NUM_THREADS=4` lets each PySR fit use 4 cores while joblib
runs roughly 19 reps in parallel. If you see OOM errors, drop N_JOBS
below.


## 1. Setup

In [ ]:
import os
import sys
import pickle
import warnings
import datetime
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '.')

from dgp import DGP_REGISTRY
from estimators import ESTIMATOR_REGISTRY, SR_PDS_LOG, PYSR_CONFIG
from simulation import run_all, run_simulation, seed_robustness_check
from evaluate import (evaluate_all, summary_table, print_summary,
                      evaluate_recovery)
from plots import (plot_bias, plot_coverage, plot_rmse_heatmap,
                   plot_summary_panel, plot_distributions,
                   plot_recovery_heatmap, plot_coefficient_recovery,
                   plot_n_trajectory, plot_recovery_vs_n)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

N_HEADLINE     = 500
P              = 50
S              = 6
BETA0          = 0.5
N_REPS_MAIN    = 500    # headline
N_REPS_CF      = 100    # cross-fit (5x slower per rep)
N_REPS_TRAJ    = 25     # trajectory + large-n
N_REPS_SEEDS   = 10     # PySR seeds for the seed-robustness check
N_JOBS         = 16     # outer parallelism (conservative; ~76/4 = 19 ceiling)
RESULTS_DIR    = '../results_full'

os.makedirs(RESULTS_DIR, exist_ok=True)
DGP_KEYS = list(DGP_REGISTRY.keys())
EST_KEYS = list(ESTIMATOR_REGISTRY.keys())

print(f"DGPs ({len(DGP_KEYS)}):       {DGP_KEYS}")
print(f"Estimators ({len(EST_KEYS)}): {EST_KEYS}")
print(f"Headline n={N_HEADLINE}, n_reps={N_REPS_MAIN} (sr_pds_cf at {N_REPS_CF})")
print(f"PySR: maxsize={PYSR_CONFIG['maxsize']}, "
      f"niter={PYSR_CONFIG['niterations']}, "
      f"parsimony=({PYSR_CONFIG['parsimony_y']}, {PYSR_CONFIG['parsimony_d']})")

---
## 2. Headline simulation

All 8 DGPs x 7 estimators at n=500. SR-PDS-CF reps are reduced to 100
via `reps_overrides`. Expected runtime ~2-3 hours on 76 cores.

In [ ]:
SR_PDS_LOG.clear()

t0 = time.time()
combined = run_all(
    dgp_registry       = DGP_REGISTRY,
    estimator_registry = ESTIMATOR_REGISTRY,
    dgp_keys           = DGP_KEYS,
    estimator_keys     = EST_KEYS,
    n                  = N_HEADLINE,
    p                  = P,
    s                  = S,
    beta0              = BETA0,
    n_reps             = N_REPS_MAIN,
    n_jobs             = N_JOBS,
    save_dir           = RESULTS_DIR,
    reps_overrides     = {'sr_pds_cf': N_REPS_CF},
)
print(f"\nHeadline complete in {(time.time()-t0)/60:.1f} min")

combined.to_pickle(f'{RESULTS_DIR}/headline.pkl')
with open(f'{RESULTS_DIR}/sr_pds_log_headline.pkl', 'wb') as f:
    pickle.dump(list(SR_PDS_LOG), f)

---
## 3. Headline metrics and plots

In [ ]:
all_metrics = evaluate_all(combined, beta0=BETA0)
all_metrics.to_csv(f'{RESULTS_DIR}/headline_metrics.csv', index=False)
print_summary(all_metrics, BETA0)

In [ ]:
plot_bias(all_metrics, save=True);          plt.show()
plot_coverage(all_metrics, save=True);      plt.show()
plot_rmse_heatmap(all_metrics, save=True);  plt.show()
plot_summary_panel(all_metrics, save=True); plt.show()
for dgp_key in DGP_KEYS:
    plot_distributions(combined, BETA0, dgp_key, save=True)
    plt.show()

---
## 4. Recovery analysis

What did SR-PDS actually discover? For each true nonlinear term in
each DGP we measure (a) how often PySR found it pre-LASSO and (b) how
often it survived the post-LASSO selection step.

In [ ]:
with open(f'{RESULTS_DIR}/sr_pds_log_headline.pkl', 'rb') as f:
    headline_log = pickle.load(f)

recovery_df = evaluate_recovery(headline_log, DGP_REGISTRY)
recovery_df.to_csv(f'{RESULTS_DIR}/recovery.csv', index=False)
print(f"Recovery records: {len(recovery_df)}")
recovery_df.head(20)

In [ ]:
if not recovery_df.empty:
    plot_recovery_heatmap(recovery_df, equation='y', save=True); plt.show()
    sub_d = recovery_df[recovery_df['equation'] == 'd']
    if not sub_d.empty:
        plot_recovery_heatmap(recovery_df, equation='d', save=True); plt.show()
    plot_coefficient_recovery(recovery_df, equation='y', save=True); plt.show()

---
## 5. N-trajectory

How do bias and RMSE evolve as n grows? Trajectory at n in {30, 40,
..., 100, 200, 500, 1000, 2000, 5000}. SR-PDS-CF is excluded (lives at
n=500 only).

Trajectory is checkpointed every 50 (DGP, estimator, n) cells to a
partial pickle so that compute crashes mid-run don't lose everything.

In [ ]:
N_GRID_TRAJ   = [30, 40, 50, 60, 70, 80, 90, 100, 200, 500, 1000, 2000, 5000]
TRAJ_EST_KEYS = [k for k in EST_KEYS if k != 'sr_pds_cf']

SR_PDS_LOG.clear()
traj_rows = []
total = len(N_GRID_TRAJ) * len(DGP_KEYS) * len(TRAJ_EST_KEYS)
i = 0
t_traj_start = time.time()

for n_grid in N_GRID_TRAJ:
    for dgp_key in DGP_KEYS:
        for est_key in TRAJ_EST_KEYS:
            i += 1
            est = ESTIMATOR_REGISTRY[est_key]
            elapsed = (time.time() - t_traj_start) / 60
            print(f"[{i:>4}/{total}] {elapsed:5.1f}min  "
                  f"n={n_grid:>5}  {dgp_key}  {est_key}", end='  ')

            df = run_simulation(
                DGP_REGISTRY[dgp_key]['fn'], est['fn'],
                estimator_key=est_key,
                n=n_grid, p=P, s=S, beta0=BETA0,
                n_reps=N_REPS_TRAJ,
                n_jobs=1 if est['requires_serial'] else N_JOBS,
                dgp_key=dgp_key,
            )
            df['dgp'] = dgp_key
            df['estimator'] = est_key
            df['dgp_label'] = DGP_REGISTRY[dgp_key]['label']
            df['est_label'] = est['label']
            df['beta0'] = BETA0
            df['n_grid'] = n_grid
            traj_rows.append(df)
            print(f"  beta={df['beta_hat'].mean():+.3f}  "
                  f"({df['failed'].sum()} failed)")

            if i % 50 == 0:
                pd.concat(traj_rows, ignore_index=True).to_pickle(
                    f'{RESULTS_DIR}/trajectory_partial.pkl')

traj_df = pd.concat(traj_rows, ignore_index=True)
traj_df.to_pickle(f'{RESULTS_DIR}/trajectory.pkl')
with open(f'{RESULTS_DIR}/sr_pds_log_traj.pkl', 'wb') as f:
    pickle.dump(list(SR_PDS_LOG), f)
print(f"\nTrajectory complete in {(time.time()-t_traj_start)/60:.1f} min")

---
## 6. Large-n extension

PySR runtime scales roughly linearly with n. We extend to n=10,000 on
DGPs 1, 6, 7, 8 and to n=50,000 only on DGPs 6, 7, 8 -- the cases
where the structural-failure story matters most. DGP-1 at n=10,000
acts as an "easy case still works" check.

Expected runtime: ~1-2 hours on 76 cores.

In [ ]:
LARGE_N_GRID = {
    10000: ['dgp1', 'dgp6', 'dgp7', 'dgp8'],
    50000: ['dgp6', 'dgp7', 'dgp8'],
}
LARGE_EST_KEYS = [k for k in EST_KEYS if k != 'sr_pds_cf']

SR_PDS_LOG.clear()
large_rows = []
total = sum(len(dgps) for dgps in LARGE_N_GRID.values()) * len(LARGE_EST_KEYS)
i = 0
t_large_start = time.time()

for n_grid, dgp_list in LARGE_N_GRID.items():
    for dgp_key in dgp_list:
        for est_key in LARGE_EST_KEYS:
            i += 1
            est = ESTIMATOR_REGISTRY[est_key]
            elapsed = (time.time() - t_large_start) / 60
            print(f"[{i:>3}/{total}] {elapsed:5.1f}min  "
                  f"n={n_grid:>6}  {dgp_key}  {est_key}")

            df = run_simulation(
                DGP_REGISTRY[dgp_key]['fn'], est['fn'],
                estimator_key=est_key,
                n=n_grid, p=P, s=S, beta0=BETA0,
                n_reps=N_REPS_TRAJ,
                n_jobs=1 if est['requires_serial'] else N_JOBS,
                dgp_key=dgp_key,
            )
            df['dgp'] = dgp_key
            df['estimator'] = est_key
            df['dgp_label'] = DGP_REGISTRY[dgp_key]['label']
            df['est_label'] = est['label']
            df['beta0'] = BETA0
            df['n_grid'] = n_grid
            large_rows.append(df)

large_df = pd.concat(large_rows, ignore_index=True)
large_df.to_pickle(f'{RESULTS_DIR}/large_n.pkl')
with open(f'{RESULTS_DIR}/sr_pds_log_large.pkl', 'wb') as f:
    pickle.dump(list(SR_PDS_LOG), f)
print(f"\nLarge-n complete in {(time.time()-t_large_start)/60:.1f} min")

In [ ]:
full_traj_df = pd.concat([traj_df, large_df], ignore_index=True)
full_traj_df.to_pickle(f'{RESULTS_DIR}/trajectory_full.pkl')
print(f"Full trajectory: {len(full_traj_df)} rows across "
      f"{full_traj_df['n_grid'].nunique()} n values")

In [ ]:
plot_n_trajectory(full_traj_df, BETA0, save=True);          plt.show()
plot_recovery_vs_n(full_traj_df, DGP_REGISTRY, save=True); plt.show()

---
## 7. Seed-robustness check

Run SR-PDS ten times on one fixed DGP-6 dataset (n=500) with different
PySR random seeds. Reports the discovered expression and beta_hat for
each seed. This backs up the reproducibility claim in the dissertation
-- if every seed finds the same modal expression, seed-dependence is
weak in this regime.

In [ ]:
seed_df = seed_robustness_check(
    dgp_fn   = DGP_REGISTRY['dgp6']['fn'],
    dgp_key  = 'dgp6',
    n        = 500, p=P, s=S, beta0=BETA0,
    data_seed   = 42,
    pysr_seeds  = list(range(1, N_REPS_SEEDS + 1)),
    save_dir = RESULTS_DIR,
)

print("\nSeed robustness summary:")
print(f"  beta_hat: mean={seed_df['beta_hat'].mean():.4f}  "
      f"std={seed_df['beta_hat'].std():.4f}  "
      f"min={seed_df['beta_hat'].min():.4f}  "
      f"max={seed_df['beta_hat'].max():.4f}")

# count how many seeds picked each distinct post-LASSO term set
term_counts = seed_df['post_lasso_terms_y'].value_counts()
print(f"\nDistinct post-LASSO y-term sets across {N_REPS_SEEDS} seeds:")
for terms, count in term_counts.items():
    print(f"  {count} seed(s): {terms}")

seed_df

---
## 8. Summary and save

In [ ]:
tag = datetime.datetime.now().strftime('%Y%m%d_%H%M')
print(f"Run tag: {tag}\n")
print(f"Files in {RESULTS_DIR}:")
for f in sorted(os.listdir(RESULTS_DIR)):
    size_mb = os.path.getsize(f'{RESULTS_DIR}/{f}') / 1024 ** 2
    print(f"  {f:<40}  {size_mb:>7.2f} MB")

In [ ]:
print('=' * 70)
print(f"  SR-PDS simulation study -- full results ({tag})")
print('=' * 70)
print(f"\n  Headline:   {N_REPS_MAIN} reps x {len(DGP_KEYS)} DGPs x "
      f"{len(EST_KEYS)} methods at n={N_HEADLINE}")
print(f"               (sr_pds_cf reduced to {N_REPS_CF} reps)")
print(f"  Trajectory: {N_REPS_TRAJ} reps x {len(DGP_KEYS)} DGPs x "
      f"{len(TRAJ_EST_KEYS)} methods at {len(N_GRID_TRAJ)} n values")
print(f"  Large-n:    {N_REPS_TRAJ} reps at n=10,000 (4 DGPs) and "
      f"n=50,000 (3 DGPs)")
print(f"  Seeds:      {N_REPS_SEEDS} PySR seeds on DGP-6 at n={N_HEADLINE}")
print(f"\n  Output:     {RESULTS_DIR}/")
print('=' * 70)